# MCXN9xx External Signed + Encrypted Image (IPED)

This notebook describes how to set up a signed **and encrypted** execute-in-place (XIP) image in **external FlexSPI flash** on an MCXN9xx device (e.g. `mcxn946`/`mcxn947`) using **IPED** (In-line PRINCE Encryption/Decryption) together with the SPSDK *nxpimage* and *pfr* applications.

IPED encrypts a region of the external FlexSPI flash on the fly: the image stored externally is a normal **signed** Master Boot Image, and the FlexSPI controller transparently decrypts the configured region during execution. The encryption region is defined in the **CMPA** (`IPED0_START`/`IPED0_END`), so no separate encrypted-image artifact is produced &mdash; confidentiality is a property of the external-flash region.

The flow is:
1. establish the Root of Trust keys and a certificate block,
2. build a signed MBI targeting the external FlexSPI XIP address,
3. generate the CMPA (with an IPED region enabled) and CFPA binaries,
4. configure the external FlexSPI flash and write CMPA/CFPA and the signed image; IPED decrypts the region on the fly during boot.

<div class="alert alert-block alert-warning">
<b>WARNING:</b> The keys and configuration used in this example are meant only for demonstration purposes and must never be used in production. Enabling IPED and writing/locking the CMPA on a real device can permanently restrict it. The external memory configuration word depends on the specific FlexSPI flash on your board.
</div>

## 1. Prerequisites

- SPSDK is needed with examples extension. `pip install spsdk[examples]` (Please refer to the [installation](../../_knowledge_base/installation_guide.rst) documentation.)
- This example uses an MCXN9xx board (e.g. FRDM-MCXN947) with an external FlexSPI flash, connected via USB (ISP mode) or UART.

<img src="../../_data/img/boards/frdm-mcxn947.png" alt="frdm-mcxn947" width="400">

In [1]:
# Initialization cell
from spsdk.utils.jupyter_utils import YamlDiffWidget

# This env variable sets colored logger output to STDOUT
%env JUPYTER_SPSDK=1
# Set a magic for command execution and echo
%alias execute echo %l && %l
%alias_magic ! execute

env: JUPYTER_SPSDK=1
Created `%!` as an alias for `%execute`.


In [2]:
FAMILY = "mcxn946"  # change this to your MCXN9xx family (e.g. mcxn947)
WORKSPACE = "workspace/"  # change this to path to your workspace
INPUTS = "inputs/"  # input configurations and binaries
KEYS = "../../_data/keys/ecc256/"  # shared example keys
# External FlexSPI flash is mapped at 0x80000000; the encrypted image starts at 0x80001000.
EXTERNAL_IMAGE_ADDRESS = "0x80001000"
VERBOSITY = (
    ""  # verbosity of commands, might be -v or -vv for debug or blank for no additional info
)

## 2. Root of Trust Keys and Certificate Block

Secure boot is anchored by up to four Root of Trust (RoT) key-pairs whose table hash (RKTH) is programmed into the CMPA. In this example we reuse the shared ECC-256 example key-pairs from the `_data/keys/ecc256` folder; in a real scenario you would generate your own keys using the *nxpcrypto* application (refer to [How-to-get-keys-using-nxpcrypto](../../crypto/keys.ipynb)).

The certificate block is provided in [cert_block_mcxn9xx.yaml](./inputs/cert_block_mcxn9xx.yaml). See the [How-to-get-cert-block](../../certificate_block/cert_block_v2_1.ipynb) notebook for details.

## 3. Generate the signed Master Boot Image (external XIP)

The image is a signed MBI whose execution address points to the external FlexSPI region (`0x80001000`); IPED handles encryption at the flash level. Below we build the signed MBI from the ready-made configuration ([mbi_config.yaml](./inputs/mbi_config.yaml)).

In [3]:
import os

MBI_CONFIG = INPUTS + "mbi_config.yaml"
%! nxpimage $VERBOSITY mbi export -c $MBI_CONFIG
assert os.path.exists(WORKSPACE + "mcxn9xx_ext_signed.bin")

nxpimage  mbi export -c inputs/mbi_config.yaml 
  ======================================== TrustZone Configuration (Warning) =========================================
  Validates TrustZone settings and configuration type
    Feature 'tz' verification(Warning) 
      Implementation(Warning): Not implemented
 (1738ms since start, mbi.py:689)
RKTH: e2cca7cf09a45d2f1942969fda1c68ecaad78fad416d143292dad2f618291ddd
Success. (Master Boot Image: workspace/mcxn9xx_ext_signed.bin created.)


## 4. Generate CMPA (with IPED enabled) and CFPA

IPED is enabled through the CMPA `IPED0_START`/`IPED0_END` registers, which define the encrypted external-flash region and its mode. Here region 0 covers `0x80001000..0x80010000` in CTR mode; the start address matches the MBI execution address. The diff below highlights the change against the default CMPA template.

In [4]:
# Get difference of template and user CMPA configuration
YamlDiffWidget("inputs/mcxn9xx_iped.diffc").html

pfr get-template -f mcxn946 -t cmpa -o workspace/cmpa_template.yaml --force 
The PFR cmpa template for mcxn946, Revision: latest has been saved into workspace/cmpa_template.yaml YAML file


In [5]:
import os

CMPA_CONFIG = INPUTS + "cmpa_mcxn9xx.yaml"
CFPA_CONFIG = INPUTS + "cfpa_mcxn9xx.yaml"
CERT_BLOCK_CONFIG = INPUTS + "cert_block_mcxn9xx.yaml"
CMPA_BIN = WORKSPACE + "cmpa.bin"
CFPA_BIN = WORKSPACE + "cfpa.bin"

%! pfr $VERBOSITY export -c $CMPA_CONFIG -e $CERT_BLOCK_CONFIG -o $CMPA_BIN
%! pfr $VERBOSITY export -c $CFPA_CONFIG -o $CFPA_BIN
assert os.path.exists(CMPA_BIN)
assert os.path.exists(CFPA_BIN)

pfr  export -c inputs/cmpa_mcxn9xx.yaml -e inputs/cert_block_mcxn9xx.yaml -o workspace/cmpa.bin 
Success. (CMPA binary has been generated and stored into workspace/cmpa.bin)
pfr  export -c inputs/cfpa_mcxn9xx.yaml -o workspace/cfpa.bin 
Success. (CFPA binary has been generated and stored into workspace/cfpa.bin)


## 5. Device preparation

Put the board into In-System Programming (ISP) mode and use *nxpdevscan* to check that the device is detected by the PC.

In [6]:
# check if the device is connected and detected by the PC
%! nxpdevscan

nxpdevscan 
Scanning USB devices...
╔══════════════════════════════════════════════════════════════════════════════════════════════════════════════════════╗
║                                        Connected NXP USB Devices  (1 device)                                         ║
╠══════════════════════════════════════════════════════════════════════════════════════════════════════════════════════╣
║  VID       : 0x1fc9                                                                                                  ║
║  PID       : 0x014f                                                                                                  ║
║  Product   : USB COMPOSITE DEVICE                                                                                    ║
║  Name      : mcxw70ad | mcxn556s | mcxn546 | mcxn557s | mcxn537t | mcxn536t | mcxn247 | mcxn527t | mcxn526 | kw43l43z║
║  Path      : HID\VID_1FC9&PID_014F\6&28545052&0&0000                                                               

In [7]:
# Select the interface to the target. USB (default MCXN9xx ISP VID:PID) or UART.
USB = "-u 0x1fc9,0x014f"  # change to your device, or use e.g. "-p com10" for UART

## 6. Configure external flash, write CMPA/CFPA and the encrypted image

First write the CFPA and CMPA (which enables IPED). Then configure the external FlexSPI flash: a memory configuration block is written to RAM and applied with `configure-memory` (memory ID `0x9` = FlexSPI). Finally erase the external region and program the signed image; IPED encrypts the region on the fly. The exact FlexSPI configuration word (here `0xc0000405`) depends on the flash device on your board &mdash; adjust it to match your hardware.

In [8]:
SIGNED_IMAGE = WORKSPACE + "mcxn9xx_ext_signed.bin"
# Write device configuration areas (CMPA enables IPED)
%! pfr $VERBOSITY write $USB -t cfpa -f $FAMILY -bin $CFPA_BIN
%! pfr $VERBOSITY write $USB -t cmpa -f $FAMILY -bin $CMPA_BIN
# Configure the external FlexSPI flash (memory ID 0x9). Adjust the config word for your flash device.
%! blhost $USB -- fill-memory 0x20000000 4 0xc0000405
%! blhost $USB -- configure-memory 0x9 0x20000000
# Erase the external region and program the signed image (IPED encrypts on the fly)
%! blhost $USB flash-erase-region $EXTERNAL_IMAGE_ADDRESS 0x10000 9
%! blhost $USB write-memory $EXTERNAL_IMAGE_ADDRESS $SIGNED_IMAGE 9
%! blhost $USB reset

pfr  write -u 0x1fc9,0x014f -t cfpa -f mcxn946 -bin workspace/cfpa.bin 
CFPA data written to device.
pfr  write -u 0x1fc9,0x014f -t cmpa -f mcxn946 -bin workspace/cmpa.bin 
CMPA data written to device.
blhost -u 0x1fc9,0x014f -- fill-memory 0x20000000 4 0xc0000405 
Response status = 0 (0x0) Success.
blhost -u 0x1fc9,0x014f -- configure-memory 0x9 0x20000000 
Response status = 0 (0x0) Success.
blhost -u 0x1fc9,0x014f flash-erase-region 0x80001000 0x10000 9 
Response status = 0 (0x0) Success.
blhost -u 0x1fc9,0x014f write-memory 0x80001000 workspace/mcxn9xx_ext_signed.bin 9 
Response status = 0 (0x0) Success.
Response word 1 = 4368 (0x1110)
blhost -u 0x1fc9,0x014f reset 
Response status = 0 (0x0) Success.
